# SignSight — MTSD: kiểm tra annotation trước

Chạy CPU. Lấy bộ **fully annotated** từ https://www.mapillary.com/dataset/trafficsign
(trang hiện yêu cầu đăng nhập). Trước mắt chỉ cần ZIP annotation có annotations/*.json và splits/train.txt.
Đây là MTSD, không phải Mapillary Vistas hoặc dữ liệu điểm biển từ Mapillary API.
Chưa cần tải mọi ZIP ảnh lớn. Không mount Drive; dùng gdown với file bạn chia sẻ, hoặc ZIP đã có trong runtime.

Notebook xuất thống kê nhãn và danh sách ảnh train ứng viên cho biển tốc độ 40/50/60/70/80.
Tên nhãn chỉ là gợi ý mapping: phải xem ảnh để xác nhận hình dạng/đơn vị và mọi biển CAR35 khác.
Không train chỉ 5 lớp thay model 35 lớp, không tự coi reviewed=yes và không thay checkpoint trong app.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown==5.2.0'])


In [ ]:
from pathlib import Path
WORK = Path('/content/signsight_mtsd')
WORK.mkdir(parents=True, exist_ok=True)
ANNOTATIONS_URL = ''  # Link gdown ZIP annotation fully annotated bạn đã tải từ MTSD
LOCAL_ANNOTATIONS_ZIP = ''  # Hoặc đường dẫn ZIP đã có trong runtime
ANNOTATIONS_SHA256 = ''  # Tùy chọn; không dùng SHA của CAR35/SupSign


In [ ]:
import sys, json, hashlib, csv, shutil
SOURCES = {'training/__init__.py': '"""Offline datasets, experiments and notebook tooling; never imported by the API."""\n', 'training/data/__init__.py': '', 'training/data/mtsd.py': '"""Annotation-only MTSD reconnaissance; no images downloaded or labels approved."""\nfrom collections import Counter\nimport json\nfrom pathlib import PurePosixPath\nimport re\nfrom zipfile import ZipFile\n\n\ndef inspect_mtsd(source):\n    """Read the provider\'s annotation ZIP before acquiring large image archives.\n\n    Only exact g1 maximum-speed labels are proposed, never approved mappings.\n    Other variants/units and annotations on the same image require visual review.\n    Train membership must come from the supplied official split lists.\n    """\n    labels, image_counts, candidates = Counter(), Counter(), []\n    explicit_partial = 0\n    schema = None\n    annotation_count = 0\n    with ZipFile(source) as archive:\n        files = [m for m in archive.infolist() if not m.is_dir()]\n        names = [m.filename for m in files]\n        if len(set(names)) != len(names):\n            raise ValueError(\'Duplicate ZIP members\')\n        splits = {}\n        for split in (\'train\', \'val\', \'test\'):\n            matches = [m for m in files if PurePosixPath(m.filename).name == split + \'.txt\']\n            if len(matches) > 1:\n                raise ValueError(\'Ambiguous split list: \' + split + \'.txt\')\n            if not matches:\n                if split == \'train\':\n                    raise ValueError(\'Missing official train.txt in annotation ZIP; supply the provider annotation/splits bundle\')\n                splits[split] = set()\n                continue\n            if matches[0].file_size > 8 * 1024**2:\n                raise ValueError(\'Oversized split list\')\n            ids = archive.read(matches[0]).decode(\'utf-8-sig\').splitlines()\n            splits[split] = {PurePosixPath(key.strip()).stem for key in ids if key.strip()}\n        if any(splits[a] & splits[b] for a, b in ((\'train\', \'val\'), (\'train\', \'test\'), (\'val\', \'test\'))):\n            raise ValueError(\'Official split lists overlap\')\n        seen_ids = set()\n        for member in sorted(files, key=lambda m: m.filename):\n            if PurePosixPath(member.filename).suffix.lower() != \'.json\':\n                continue\n            if member.file_size > 16 * 1024**2:\n                raise ValueError(\'Oversized annotation: \' + member.filename)\n            content = json.loads(archive.read(member))\n            if not isinstance(content, dict) or \'objects\' not in content:\n                continue  # E.g. provider class/config metadata.\n            objects = content[\'objects\']\n            if not isinstance(objects, list) or any(not isinstance(o, dict) or not isinstance(o.get(\'label\'), str) for o in objects):\n                raise ValueError(\'Unexpected MTSD objects schema: \' + member.filename)\n            key = PurePosixPath(member.filename).stem\n            if key in seen_ids:\n                raise ValueError(\'Duplicate annotation image key: \' + key)\n            seen_ids.add(key)\n            annotation_count += 1\n            if schema is None:\n                schema = dict(annotation_keys=sorted(content), object_keys=sorted(objects[0]) if objects else [])\n            counts = Counter(o[\'label\'] for o in objects)\n            labels.update(counts)\n            image_counts.update(counts.keys())\n            partial = content.get(\'is_fully_annotated\') is False\n            explicit_partial += partial\n            wanted = sorted(label for label in counts if re.fullmatch(\n                r\'regulatory--maximum-speed-limit-(40|50|60|70|80)--g1\', label))\n            if key in splits[\'train\'] and wanted and not partial:\n                candidates.append(dict(image_key=key, annotation=member.filename,\n                    source_split=\'train\', proposed_labels=\' | \'.join(wanted),\n                    all_labels=\' | \'.join(sorted(counts)), reviewed=\'no\'))\n    if not annotation_count:\n        raise ValueError(\'No MTSD object annotations found; use the fully annotated annotation ZIP\')\n    mappings = [dict(source_label=label, target=\'Speed_limit_\' + label.split(\'--\')[1].split(\'-\')[-1], reviewed=\'no\')\n                for label in sorted(labels) if re.fullmatch(r\'regulatory--maximum-speed-limit-(40|50|60|70|80)--g1\', label)]\n    return dict(summary=dict(annotation_images=annotation_count, source_classes=len(labels),\n        split_list_images={s: len(ids) for s, ids in splits.items()},\n        explicitly_partial_images=explicit_partial, train_candidate_images=len(candidates),\n        automatic_training_ready=False,\n        limitation=\'Annotation-only scan: fully annotated archive provenance, unit/appearance, all CAR35 boxes, image integrity and exact/near holdout overlap still require review.\'),\n        class_counts=[dict(source_label=k, boxes=labels[k], images=image_counts[k]) for k in sorted(labels)],\n        mapping_candidates=mappings, train_candidates=candidates, schema_example=schema)\n', 'training/data/downloads.py': '"""Download a shared Drive ZIP to local Colab storage, without account cookies."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nfrom urllib.parse import parse_qs, urlparse\nimport zipfile\n\n\nclass ZipChecksumMismatch(ValueError):\n    """Complete ZIP retained for diagnosis, never published as the expected file."""\n\n\ndef drive_file_id(source):\n    source = str(source).strip()\n    if not source or \'YOUR_\' in source or \'PASTE_\' in source:\n        raise ValueError(\'Paste a shared Google Drive file URL or file ID\')\n    if \'://\' in source:\n        parsed = urlparse(source)\n        if parsed.scheme != \'https\' or parsed.hostname != \'drive.google.com\' or parsed.username:\n            raise ValueError(\'Use a Google Drive file link, not a folder or another website\')\n        match = re.fullmatch(r\'/file/d/([A-Za-z0-9_-]+)(?:/.*)?\', parsed.path)\n        if match:\n            source = match[1]\n        elif parsed.path in (\'/uc\', \'/open\'):\n            source = parse_qs(parsed.query).get(\'id\', [\'\'])[0]\n        else:\n            raise ValueError(\'Use a Google Drive file link, not a folder\')\n    if not re.fullmatch(r\'[A-Za-z0-9_-]{10,}\', source):\n        raise ValueError(\'Invalid Google Drive file ID\')\n    return source\n\n\ndef download_drive_zip(source, target, *, expected_sha256=\'\', downloader=None):\n    """Publish only a complete ZIP; source receipts prevent resuming another file.\n\n    gdown owns its resumable temporary files. This helper keeps those on network\n    failure and never overwrites a pre-existing target from an unknown source.\n    An authoritative SHA-256 can adopt an already downloaded matching ZIP.\n    """\n    file_id = drive_file_id(source)\n    checksum = str(expected_sha256).strip().lower()\n    if checksum and not re.fullmatch(\'[a-f0-9]{64}\', checksum):\n        raise ValueError(\'Expected SHA-256 must have 64 hexadecimal characters\')\n    target = Path(target).resolve()\n    receipt = target.with_name(target.name+\'.source.json\')\n    staging = target.with_name(target.name+\'.download\')\n\n    def validate(path):\n        if not path.is_file() or not zipfile.is_zipfile(path):\n            raise ValueError(\'Download is not a ZIP. Check Drive sharing, quota and the file link.\')\n        if checksum:\n            with path.open(\'rb\') as stream:\n                actual = hashlib.file_digest(stream, \'sha256\').hexdigest()\n            if actual != checksum:\n                raise ZipChecksumMismatch(f\'Downloaded ZIP SHA-256 mismatch for {path.name}: \'\n                    f\'expected {checksum}; actual {actual}. Complete ZIP kept at {path}; check the source link/version.\')\n\n    known = receipt.exists()\n    if known:\n        try:\n            same_source = json.loads(receipt.read_text())[\'file_id\'] == file_id\n        except (ValueError, KeyError, TypeError):\n            same_source = False\n        if not same_source: raise FileExistsError(\'Local download belongs to another Drive file; choose a new path\')\n    elif target.exists():\n        if not checksum: raise FileExistsError(\'Existing ZIP has no source receipt; supply its SHA-256 or choose a new path\')\n        validate(target)\n    elif staging.exists() or any(target.parent.glob(staging.name+\'*.tmp\')):\n        raise FileExistsError(\'Unknown partial download exists; choose a new local path\')\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if not known:\n        with receipt.open(\'x\', encoding=\'utf-8\') as stream:\n            json.dump({\'file_id\': file_id}, stream)\n    if target.exists():\n        validate(target)\n        return target\n    if staging.exists():\n        try:\n            validate(staging)\n        except ZipChecksumMismatch:\n            raise\n        except ValueError:\n            staging.unlink()  # Only the staging file identified by our matching receipt.\n    if not staging.exists():\n        if downloader is None:\n            import gdown\n            downloader = gdown.download\n        try:\n            result = downloader(id=file_id, output=str(staging), quiet=False,\n                                resume=True, use_cookies=False)\n        except Exception as error:\n            raise RuntimeError(\'Drive download failed. Check public-link access/quota, then rerun this cell \'\n                               \'to resume in the same runtime.\') from error\n        if result is None:\n            raise RuntimeError(\'Drive returned no download. Check public-link access and download quota.\')\n    try:\n        validate(staging)\n    except ZipChecksumMismatch:\n        raise\n    except ValueError:\n        staging.unlink(missing_ok=True)\n        raise\n    if target.exists(): raise FileExistsError(\'Download target appeared; it was not overwritten\')\n    staging.rename(target)\n    return target\n'}
CODE_ROOT = WORK / 'pipeline'
for relative, source in SOURCES.items():
    target = CODE_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
from training.data.downloads import download_drive_zip
from training.data.mtsd import inspect_mtsd


In [ ]:
if LOCAL_ANNOTATIONS_ZIP.strip():
    source = Path(LOCAL_ANNOTATIONS_ZIP)
else:
    source = download_drive_zip(ANNOTATIONS_URL, WORK / 'annotations.zip', expected_sha256=ANNOTATIONS_SHA256)
with source.open('rb') as stream:
    actual_sha = hashlib.file_digest(stream, 'sha256').hexdigest()
if ANNOTATIONS_SHA256.strip():
    assert actual_sha == ANNOTATIONS_SHA256.strip().lower(), 'Annotation ZIP checksum khác.'
report = inspect_mtsd(source)
report['source_zip_sha256'] = actual_sha
print(json.dumps(report['summary'], ensure_ascii=False, indent=2))
print('Schema:', report['schema_example'])
print('Mapping ứng viên:', report['mapping_candidates'])


In [ ]:
from datetime import datetime, timezone
output = WORK / ('mtsd_inspect_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
output.mkdir()
(output / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
for key in ('class_counts', 'mapping_candidates', 'train_candidates'):
    rows = report[key]
    if rows:
        with (output / (key + '.csv')).open('w', newline='', encoding='utf-8-sig') as stream:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
            writer.writeheader(); writer.writerows(rows)
result = shutil.make_archive(str(output), 'zip', output)
print('Gửi ZIP báo cáo này trước khi tải/ghép ảnh:', result)
from google.colab import files
files.download(result)


## Sau khi đọc báo cáo

Lấy đúng các image archive train theo bộ MTSD được cấp. Bước sau mới chọn khoảng 500–1.500 ảnh
(mục tiêu pilot, phụ thuộc dữ liệu thực), xem ảnh/mapping, đổi toàn bộ nhãn phù hợp sang ID CAR35 0–34,
loại ảnh trùng/gần trùng với tất cả split Việt Nam. Giữ nguyên val/test Việt Nam và các nhóm cảnh liên quan.
Ảnh MTSD val/test không được lấy làm train trong pilot này.

Ghép với dữ liệu train Việt Nam hiện có, fine-tune từ checkpoint đã chốt rồi so cùng baseline trên cùng validation
và ảnh Việt Nam riêng. Không coi mAP MTSD là chất lượng chạy tại Việt Nam. /content mất khi runtime bị xóa.
